# Stress-Predict Dataset Assignment
**Name:** Jack Hickey <br>
**Student ID:** 26257770

### 1. Understanding the Dataset

The data presented is a timeseries data with a Participant ID, Heart Rate, labelled HR, and Respiration, labelled respr, the Time in seconds, and a label to indicate stressed(1) or not stressed(0). This is clearly falls under supervised learning, specifically binary classification, as it's about predicting distinct labels(0 or 1). A regression task would be about predicting a number in a series of values. The values here while numerical are just a label which can be either 0 or 1. 

The problem being asked is to classify whether the heart rate and respiration data indicate stress or not. Each training instance provides two indepedent feature inputs (HR and Respiration) and a known ground truth dependent target. 

A potential challenge with this dataset based on first glance is that different participants could have varying levels of resting respiration and resting heart rate which could mean for a very healthy person there stressed rates could be similar to those of someone who is less healthy and not stressed. 

### 2. Data Exploration

The dataset comprises 112,516 physiological recordings across 34 distinct participants. To prevent data leakage caused by correlating sequential time-series entries from the same individual, the train/test split was performed at the participant level rather than through naive row shuffling. Applying an 80/20 subject split allocates 27 participants to the training set and 7 participants to the test set:
* **Training samples:** 90,071 rows (~80.05%)
* **Testing samples:** 22,445 rows (~19.95%)

Inspection of feature integrity reveals that `respr`, `Time(sec)`, and `Label` contain complete records (112,516 non-null entries), while `HR` contains 44 missing values (112,472 non-null entries) that require imputation or filtering during preprocessing.

Analysis of the target variable reveals moderate class imbalance:
* **Class 0 (Non-stressed):** ~67.28%
* **Class 1 (Stressed):** ~32.72%

Because the majority class accounts for roughly two-thirds of the observations, a trivial majority-class classifier would achieve ~67.3% baseline accuracy. Consequently, raw accuracy is insufficient on its own, and model evaluation must incorporate Precision, Recall, and F1-score to measure performance on the minority stress class.

In [1]:
import pandas as pd

data = pd.read_csv("data/Data.csv")

data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112516 entries, 0 to 112515
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Participant  112516 non-null  int64  
 1   HR           112472 non-null  float64
 2   respr        112516 non-null  float64
 3   Time(sec)    112516 non-null  int64  
 4   Label        112516 non-null  int64  
dtypes: float64(2), int64(3)
memory usage: 4.3 MB


In [2]:
# df['Label'].value_counts()
data['Label'].value_counts(normalize=True) * 100

Label
0    67.280209
1    32.719791
Name: proportion, dtype: float64

In [3]:
participants = data['Participant'].unique()
print(f"Partipants: {participants}")

Partipants: [ 2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25
 26 27 28 29 30 31 32 33 34 35]


In [4]:
from sklearn.model_selection import train_test_split

train_ids, test_ids = train_test_split(participants, test_size=0.20, random_state=42)

In [5]:
train_df = data[data['Participant'].isin(train_ids)]
test_df = data[data['Participant'].isin(test_ids)]

In [6]:
print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

Training rows: 90071
Testing rows: 22445



### 3. Choosing an ML Package

Several open-source packages support classification workflows across different paradigms:
* **Scikit-Learn (Python):** The standard open-source library for classical, tabular machine learning, offering tools for preprocessing, classification, regression, clustering, and model validation.
* **XGBoost / LightGBM:** Highly optimized libraries designed specifically for gradient-boosted decision trees, widely used for structured tabular classification.
* **PyTorch / TensorFlow:** Deep learning frameworks designed for high-dimensional, unstructured data (images, audio, complex time series) requiring GPU acceleration.
* **WEKA:** A widely used academic suite offering GUI- and Java-based implementations of classical learners like J48 (C4.5 decision trees).

#### Selected Package: Scikit-Learn
Scikit-Learn is selected as the primary library for this assignment due to several key features:
1. **Unified and Consistent API:** Models and transformers adhere to a standard interface (`fit()`, `predict()`, and `transform()`), simplifying comparisons between different classifiers.
2. **Integrated End-to-End Workflow:** It provides native utilities for all stages of the CRISP-DM lifecycle, including participant splitting (`model_selection`), data transformation and scaling (`preprocessing`), and algorithmic modeling (`tree`, `neighbors`).
3. **Comprehensive Metric Suite:** Built-in functions like `classification_report`, `confusion_matrix`, and `f1_score` allow direct handling and evaluation of moderately imbalanced classification problems without external dependencies.
4. **Computational Efficiency for Tabular Data:** Because the dataset consists of tabular physiological signals rather than high-dimensional image tensors, Scikit-Learn’s CPU-optimized implementations offer fast convergence without the overhead of deep learning architectures.

### 4. Data Pre-processing

Examination of the dataset reveals 44 missing values within the Heart Rate (`HR`) column. While mean or median imputation is a common strategy, injecting static values into continuous physiological time-series data creates artificial flatlines. Distance-based algorithms (like KNN) and threshold-based algorithms (like Decision Trees) can misinterpret these artifacts. Given that 44 rows constitute a negligible fraction of the 112,516 total entries, dropping these rows entirely is the safest approach to maintain data integrity.

Feature selection is critical to ensure the model generalizes to unseen individuals. The `Participant` column is removed from the feature set ($X$); retaining it would allow the algorithm to associate a specific participant ID directly with a stress state, leading to severe overfitting. The `Time(sec)` column is also dropped, as it serves merely as a temporal index and provides no independent predictive value for identifying stress states.

To prepare the remaining physiological features (`HR` and `respr`) for distance-based algorithms such as KNN, feature scaling is required. Distance metrics (such as Euclidean or Manhattan) evaluate all dimensions equally; therefore, without scaling, Heart Rate (which has a naturally larger numerical range) would mathematically dominate Respiration during distance calculations. To resolve this, we apply Z-Normalisation, which rescales each dimension independently so that it has a mean of 0 and a standard deviation of 1 using the formula $D \leftarrow (D - Mean) / StDev$. This mathematical transformation is implemented via Scikit-Learn's `StandardScaler`. To rigorously prevent data leakage, the scaler is fitted exclusively on the training partition to calculate the baseline mean and standard deviation. These learned parameters are then applied to transform both the training and testing sets independently.


### 5. Algorithm Selection and Application
*Your rationale for choosing your two algorithms (e.g., Decision Trees and KNN) and the implementation details.*

### 6. Model Evaluation
*Your performance metrics (Accuracy, Precision, Recall, F1) and visualizations.*

### 7. Comparative Analysis
*Your critical reflection on the strengths, weaknesses, and performance differences between the two models.*